In [ ]:
import os
import subprocess

# 1. SETUP PERCORSI
PATH_OMNI_DIR = '/kaggle/input/omnivore-features/omnivore'

# 2. CLONAZIONE ANNOTAZIONI (se mancano)
REPO_DIR = '/kaggle/working/annotations'
if not os.path.exists(REPO_DIR):
    print(" Clonazione annotazioni...")
    subprocess.check_call("git clone https://github.com/CaptainCook4D/annotations.git " + REPO_DIR, shell=True)

PATH_CSV = os.path.join(REPO_DIR, 'annotation_csv', 'step_annotations.csv')

# 3. CLONAZIONE CODICE ACTIONFORMER (se manca)
if not os.path.exists('/kaggle/working/actionformer_release'):
    print(" Clonazione ActionFormer...")
    !git clone https://github.com/happyharrycn/actionformer_release.git
else:
    print(" ActionFormer già presente.")

# 4. VERIFICA FILE
def verify(directory, label):
    if os.path.exists(directory):
        print(f" {label}: {len(os.listdir(directory))} file trovati.")
    else:
        print(f" {label}: Cartella NON trovata!")

verify(PATH_OMNI_DIR, "Omnivore")

🐙 Clonazione annotazioni...


Cloning into '/kaggle/working/annotations'...


⬇️ Clonazione ActionFormer...
Cloning into 'actionformer_release'...
remote: Enumerating objects: 394, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 394 (delta 5), reused 2 (delta 2), pack-reused 385 (from 2)
Receiving objects: 100% (394/394), 637.60 KiB | 4.69 MiB/s, done.
Resolving deltas: 100% (217/217), done.
✅ Omnivore: 384 file trovati.


In [ ]:
import os
import numpy as np
import torch

# 1. SETUP PERCORSI
# https://drive.google.com/file/d/127NV-epD7MOauiMbQ2pS7Qt6KKn-Me4k/view?usp=drive_link
FILE_ID = '127NV-epD7MOauiMbQ2pS7Qt6KKn-Me4k' # <-- METTI IL TUO ID QUI
ZIP_PATH = '/kaggle/working/nuove_feature.zip'
EXTRACT_PATH = '/kaggle/working/nuove_feature'

# 2. DOWNLOAD
print(" Avvio download da Google Drive...")
!gdown {FILE_ID} -O {ZIP_PATH}

# 3. SCOMPATTAMENTO
if not os.path.exists(EXTRACT_PATH):
    os.makedirs(EXTRACT_PATH)

print(" Scompattamento in corso...")
!unzip -q -o {ZIP_PATH} -d {EXTRACT_PATH}

# Pulizia: a volte unzip crea una sottocartella con lo stesso nome, 
# cerchiamo i file .npz ricorsivamente
tutti_i_file = []
for root, dirs, files in os.walk(EXTRACT_PATH):
    for file in files:
        if file.endswith(('.npz', '.npy')):
            tutti_i_file.append(os.path.join(root, file))

print(f" Scompattamento completato. File trovati: {len(tutti_i_file)}")

# 4. TEST DIMENSIONE FEATURE
if len(tutti_i_file) > 0:
    test_file = tutti_i_file[0]
    print(f" Analisi file di test: {os.path.basename(test_file)}")
    
    try:
        data = np.load(test_file)
        # Gestione .npz (file compresso) o .npy (array semplice)
        feat = data[data.files[0]] if hasattr(data, 'files') else data
        
        shape = feat.shape
        # Se shape è (Canali, Tempo) la invertiamo per coerenza
        if shape[0] < shape[1] and shape[0] != 768:
            dim_canali = shape[0]
            lunghezza_t = shape[1]
        else:
            dim_canali = shape[1]
            lunghezza_t = shape[0]

        print(f"\n RISULTATI TEST:")
        print(f"  - Dimensione Feature (Canali): {dim_canali}")
        print(f"  - Lunghezza Temporale: {lunghezza_t}")
        
        if dim_canali == 768:
            print(" OTTIMO: La dimensione è 768 come previsto!")
        else:
            print(f" ATTENZIONE: La dimensione rilevata è {dim_canali}, non 768.")
            
    except Exception as e:
        print(f" Errore durante la lettura del file: {e}")
else:
    print(" Errore: Nessun file .npz o .npy trovato dopo l'estrazione.")

📥 Avvio download da Google Drive...
Downloading...
From (original): https://drive.google.com/uc?id=127NV-epD7MOauiMbQ2pS7Qt6KKn-Me4k
From (redirected): https://drive.google.com/uc?id=127NV-epD7MOauiMbQ2pS7Qt6KKn-Me4k&confirm=t&uuid=4c5b12cc-9e46-433d-83bd-8f7091d88ff8
To: /kaggle/working/nuove_feature.zip
100%|███████████████████████████████████████| 1.81G/1.81G [00:07<00:00, 250MB/s]
📦 Scompattamento in corso...
✅ Scompattamento completato. File trovati: 384
🔍 Analisi file di test: 26_22_360p_224_0s_1s.npz

📊 RISULTATI TEST:
  - Dimensione Feature (Canali): 768
  - Lunghezza Temporale: 2482
🚀 OTTIMO: La dimensione è 768 come previsto!


In [ ]:
import os
import sys
import importlib

# 1. PERCORSI CRITICI
ROOT_DIR = "/kaggle/working/actionformer_release"
UTILS_DIR = os.path.join(ROOT_DIR, "libs", "utils")

# 2. AGGIUNTA FORZATA AL PATH
# Diciamo a Python: "Guarda anche dentro libs/utils quando cerchi moduli!"
if UTILS_DIR not in sys.path:
    sys.path.insert(0, UTILS_DIR)
    print(f" Aggiunto {UTILS_DIR} a sys.path")

if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)
    print(f" Aggiunto {ROOT_DIR} a sys.path")

# 3. VERIFICA FILE COMPILATO
# Controlliamo se il file .so esiste davvero
files = os.listdir(UTILS_DIR)
so_files = [f for f in files if "nms_1d_cpu" in f and f.endswith(".so")]

if so_files:
    print(f" File compilato trovato: {so_files[0]}")
else:
    print(" ERRORE: Non trovo il file .so in libs/utils. La compilazione ha fallito silenziosamente?")

# 4. RESET E IMPORT
os.chdir(ROOT_DIR)
importlib.invalidate_caches()

print("\n Test finale importazione...")
try:
    # Proviamo a importare direttamente il modulo C++
    import nms_1d_cpu
    print("   -> Modulo C++ 'nms_1d_cpu' caricato! ")
    
    # Proviamo a importare il modello completo
    from libs.modeling.meta_archs import PtTransformer
    print(" SUCCESSO: PtTransformer importato e pronto all'uso!")
    
except ImportError as e:
    print(f" Errore Importazione: {e}")

🔎 File compilato trovato: nms_1d_cpu.cpython-312-x86_64-linux-gnu.so

🧪 Test finale importazione...
   -> Modulo C++ 'nms_1d_cpu' caricato! 🟢
🚀 SUCCESSO: PtTransformer importato e pronto all'uso!


In [ ]:
import os
import sys
import subprocess

# 1. Spostiamoci nella cartella dove c'è il file setup.py per le utility
# In ActionFormer di solito è in libs/utils
SETUP_DIR = "/kaggle/working/actionformer_release/libs/utils"

print(f" Avvio compilazione moduli C++ in: {SETUP_DIR}")

if os.path.exists(SETUP_DIR):
    os.chdir(SETUP_DIR)
    
    try:
        # 2. Eseguiamo il comando di build
        # build_ext --inplace crea il file compilato (.so) direttamente nella cartella corrente
        result = subprocess.check_output(["python", "setup.py", "build_ext", "--inplace"], stderr=subprocess.STDOUT)
        print(result.decode("utf-8")) # Stampa il log della compilazione
        print(" Compilazione completata con successo!")
        
    except subprocess.CalledProcessError as e:
        print(" Errore durante la compilazione:")
        print(e.output.decode("utf-8"))
else:
    print(f" Cartella non trovata: {SETUP_DIR}")

# 3. Torniamo alla root e riproviamo l'import
ROOT_DIR = "/kaggle/working/actionformer_release"
os.chdir(ROOT_DIR)
print(f"\n Directory di lavoro ripristinata: {os.getcwd()}")

# Aggiungiamo il path se manca
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print("\n Test importazione post-compilazione...")
try:
    import libs
    from libs.modeling.meta_archs import PtTransformer
    print(" SUCCESSO: PtTransformer importato e funzionante!")
except ImportError as e:
    print(f" Ancora errore: {e}")

🔨 Avvio compilazione moduli C++ in: /kaggle/working/actionformer_release/libs/utils
running build_ext
building 'nms_1d_cpu' extension
ninja: no work to do.
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 /kaggle/working/actionformer_release/libs/utils/build/temp.linux-x86_64-cpython-312/./csrc/nms_cpu.o -L/usr/local/lib/python3.12/dist-packages/torch/lib -L/usr/lib/x86_64-linux-gnu -lc10 -ltorch -ltorch_cpu -ltorch_python -o build/lib.linux-x86_64-cpython-312/nms_1d_cpu.cpython-312-x86_64-linux-gnu.so
copying build/lib.linux-x86_64-cpython-312/nms_1d_cpu.cpython-312-x86_64-linux-gnu.so -> 

✅ Compilazione completata con successo!

📂 Directory di lavoro ripristinata: /kaggle/working/actionformer_release

🧪 Test importazione post-compilazione...
🎉 SUCCESSO: PtTransformer importato e funzionante!


In [ ]:
import os
import sys
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
import random
import time
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW, lr_scheduler 

# Aggiunta path per ActionFormer
if os.path.exists('/kaggle/working/actionformer_release'):
    sys.path.append('/kaggle/working/actionformer_release')
from libs.modeling.meta_archs import PtTransformer

# ==============================================================================
# 1. DATASET AGGIORNATO (New Features 768 + Omnivore 1024 = 1792)
# ==============================================================================
print(" Configurazione Dataset con Nuove Feature (768 canali)...")

# Percorsi aggiornati
PATH_EGO_DIR = '/kaggle/working/nuove_feature' # Le nuove feature scaricate
PATH_OMNI_DIR = '/kaggle/input/omnivore-features/omnivore'
PATH_CSV = '/kaggle/working/annotations/annotation_csv/step_annotations.csv'

class SmartRandomCropDataset(Dataset):
    def __init__(self, csv_path, ego_dir, omni_dir, max_len=2304, training=True):
        self.df = pd.read_csv(csv_path)
        self.max_len = max_len
        self.training = training
        self.fps = 1.875 
        self.target_dim = 1792 # 768 + 1024 = 1792 (PERFETTO)
        
        print("🔍 Mappatura file in corso (Ricorsiva)...")
        self.ego_map = self._build_file_map_recursive(ego_dir)
        self.omni_map = self._build_file_map_recursive(omni_dir)
        
        self.df['recording_id'] = self.df['recording_id'].astype(str)
        valid_csv_ids = set(self.df['recording_id'].unique())
        common_ids = set(self.ego_map.keys()) & set(self.omni_map.keys())
        self.video_ids = list(common_ids.intersection(valid_csv_ids))
        print(f" Setup completato: {len(self.video_ids)} video pronti per il training.")

    def _build_file_map_recursive(self, directory):
        """Trova i file .npz/.npy anche in sottocartelle create dall'unzip"""
        file_map = {}
        if not os.path.exists(directory): return file_map
        for root, _, files in os.walk(directory):
            for fname in files:
                if not fname.endswith(('.npz', '.npy')): continue
                # Estrazione ID (funziona con i nomi '7_2_360p_224_0s_1s.npz')
                vid_id = fname.split('_360p')[0].split('_256')[0].replace('.mp4', '')
                if len(vid_id) > 10 and '_' in vid_id:
                     parts = vid_id.split('_')
                     if len(parts) >= 2 and parts[1].isdigit():
                         vid_id = f"{parts[0]}_{parts[1]}"
                file_map[vid_id] = os.path.join(root, fname)
        return file_map

    def __len__(self): return len(self.video_ids)

    def __getitem__(self, idx):
        vid_id = self.video_ids[idx]
        try:
            d1 = np.load(self.ego_map[vid_id])
            raw_ego = d1[d1.files[0]] if hasattr(d1, 'files') else d1
            d2 = np.load(self.omni_map[vid_id])
            raw_omni = d2[d2.files[0]] if hasattr(d2, 'files') else d2
            
            # Formattazione [Tempo, Canali]
            if raw_ego.shape[0] < raw_ego.shape[1]: raw_ego = raw_ego.T
            if raw_omni.shape[0] < raw_omni.shape[1]: raw_omni = raw_omni.T
            
            f_ego = torch.from_numpy(raw_ego).float()
            f_omni = torch.from_numpy(raw_omni).float()
            
            # Sincronizzazione temporale (taglio al minimo comune)
            min_t = min(f_ego.shape[0], f_omni.shape[0])
            # Concatenazione: 768 + 1024 = 1792
            feat = torch.cat([f_ego[:min_t], f_omni[:min_t]], dim=1)
            
            # Safety Check (ora feat.shape[1] sarà già 1792, quindi non aggiungerà zeri)
            if feat.shape[1] < self.target_dim:
                padding = torch.zeros((feat.shape[0], self.target_dim - feat.shape[1]))
                feat = torch.cat([feat, padding], dim=1)
            
            # Logica Random Crop (Invariata)
            total_frames = feat.shape[0]
            if self.training and total_frames > self.max_len:
                start_frame = random.randint(0, total_frames - self.max_len)
                feat_crop = feat[start_frame : start_frame + self.max_len, :]
                time_offset = start_frame / self.fps
            else:
                feat_crop = feat[:self.max_len, :]
                time_offset = 0.0
                if feat_crop.shape[0] < self.max_len:
                    pad = torch.zeros((self.max_len - feat_crop.shape[0], self.target_dim))
                    feat_crop = torch.cat([feat_crop, pad], dim=0)

            # Gestione Annotazioni (Invariata)
            rows = self.df[self.df['recording_id'] == vid_id]
            valid_segments = []
            labels = []
            window_dur = self.max_len / self.fps
            for _, row in rows.iterrows():
                s, e = float(row['start_time']) - time_offset, float(row['end_time']) - time_offset
                if e > 0 and s < window_dur:
                    valid_segments.append([max(0.0, s), min(window_dur, e)])
                    labels.append(0)

            return {
                'video_id': vid_id,
                'feats': feat_crop.t(), 
                'segments': torch.tensor(valid_segments).float() if valid_segments else torch.zeros((0,2)).float(),
                'labels': torch.tensor(labels).long() if labels else torch.zeros((0,)).long(),
                'fps': self.fps, 'duration': feat_crop.shape[0]/self.fps,
                'feat_stride': 1, 'feat_num_frames': feat_crop.shape[0]
            }
        except: return None

# Istanzia dataset
dataset = SmartRandomCropDataset(PATH_CSV, PATH_EGO_DIR, PATH_OMNI_DIR, training=True)

# ==============================================================================
# 2. TRAINING LOOP AGGRESSIVO (120 Epoche)
# ==============================================================================
def collate_fn_fix(batch): return [b for b in batch if b is not None]

NUM_EPOCHS = 120
BATCH_SIZE = 4
LEARNING_RATE = 1e-4 
CKPT_DIR = '/kaggle/working/checkpoints_best_features'
os.makedirs(CKPT_DIR, exist_ok=True)

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn_fix, num_workers=2, pin_memory=True)

cfg = {
    "input_dim": 1792, "num_classes": 1, "max_seq_len": 2304, 
    "backbone_type": "convTransformer", "backbone_arch": [2, 2, 5], 
    "scale_factor": 2, "n_head": 4, "n_mha_win_size": 9, 
    "embd_kernel_size": 3, "embd_dim": 512, "embd_with_ln": True, 
    "use_abs_pe": True, "use_rel_pe": False, "max_buffer_len_factor": 6.0, 
    "fpn_type": "identity", "fpn_dim": 512, "fpn_start_level": 0, "fpn_with_ln": True, 
    "head_dim": 512, "head_num_layers": 3, "head_kernel_size": 3, "head_with_ln": True, 
    "regression_range": [[0, 4], [4, 8], [8, 16], [16, 32], [32, 64], [64, 10000]], 
    "train_cfg": {
        "init_loss_norm": 200, "center_sample": "radius", "center_sample_radius": 0.7, 
        "label_smoothing": 0.0, "loss_weight": 2.0, "cls_prior_prob": 0.01, 
        "dropout": 0.1, "droppath": 0.1, "head_empty_cls": []
    },
    "test_cfg": {"pre_nms_topk": 5000, "pre_nms_thresh": 0.001, "max_seg_num": 2000, "nms_method": "hard", "iou_threshold": 0.1, "min_score": 0.001, "duration_thresh": 0.05, "multiclass_nms": True, "nms_sigma": 0.50, "voting_thresh": 0.75}
}

model = PtTransformer(**cfg).cuda()
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

# Caricamento Checkpoint Iniziale (facoltativo, se vuoi partire dai tuoi pesi migliori)
RESUME_PATH = '/kaggle/input/actionformer-best-epoch-trained-pth/actionformer_best_epoch_trained.pth'
if os.path.exists(RESUME_PATH):
    print(f" RESUME: Caricamento pesi iniziali...")
    state = torch.load(RESUME_PATH)
    model.load_state_dict({k.replace('module.', ''): v for k, v in state.items()})

print(f"\n AVVIO TRAINING (120 epoche - Dense Input 1792)...")
best_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    start_time = time.time()
    
    for i, batch in enumerate(train_loader):
        if not batch: continue
        optimizer.zero_grad()
        loss = model(batch)['final_loss']
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        if i % 20 == 0: print(f"   Ep [{epoch+1}/{NUM_EPOCHS}] Bt [{i}/{len(train_loader)}] Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / len(train_loader)
    scheduler.step()
    print(f" EPOCA {epoch+1} FINITA | Loss: {avg_loss:.4f} | Tempo: {time.time()-start_time:.1f}s")
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, "best_model_loss.pth"))
        print(f" NUOVO RECORD!")

print("\n TRAINING COMPLETATO!")

🔄 Configurazione Dataset con Nuove Feature (768 canali)...
🔍 Mappatura file in corso (Ricorsiva)...
✅ Setup completato: 384 video pronti per il training.
♻️ RESUME: Caricamento pesi iniziali...

🚀 AVVIO TRAINING (120 epoche - Dense Input 1792)...


KeyboardInterrupt: 

In [ ]:
import torch
import pandas as pd
import os
import numpy as np

# --- CONFIGURAZIONE PERCORSI ---
# Il percorso esatto dei pesi che mi hai dato
WEIGHTS_PATH = '/kaggle/input/best-model-loss-768/best_model_loss_768.pth'
OUTPUT_CSV = '/kaggle/working/inferenza_finale_768.csv'

# Percorsi Dataset (VERIFICA CHE SIANO CORRETTI)
# Se hai riavviato la sessione, assicurati di aver riscaricato/scompattato le feature
PATH_EGO_DIR = '/kaggle/working/nuove_feature' 
PATH_OMNI_DIR = '/kaggle/input/omnivore-features/omnivore'
PATH_GT_CSV = '/kaggle/working/annotations/annotation_csv/step_annotations.csv'

# --- CONFIGURAZIONE MODELLO (Soft-NMS per massimizzare la Recall) ---
# Usiamo i parametri migliori per il test
cfg_test = cfg.copy()
cfg_test['test_cfg'] = {
    "pre_nms_topk": 5000,
    "pre_nms_thresh": 0.001,
    "max_seg_num": 2000,
    "nms_method": "soft",      
    "iou_threshold": 0.5,      
    "min_score": 0.001,
    "duration_thresh": 0.05,
    "multiclass_nms": True,
    "nms_sigma": 0.50,
    "voting_thresh": 0.75
}

# --- FUNZIONE INFERENZA ---
def run_final_inference(model, dataset, output_path):
    model.eval()
    results = []
    print(f" Avvio Inferenza su {len(dataset)} video...")
    print(f" Output sarà salvato in: {output_path}")
    
    with torch.no_grad():
        for i in range(len(dataset)):
            batch = dataset[i]
            if batch is None: continue
            
            # Prepariamo input su GPU
            feats = batch['feats'].unsqueeze(0).cuda()
            
            # Dizionario COMPLETO per evitare crash
            video_input = {
                'feats': feats[0],
                'video_id': batch['video_id'],
                'fps': batch['fps'],
                'duration': batch['duration'],
                'feat_stride': batch['feat_stride'],       
                'feat_num_frames': batch['feat_num_frames']
            }
            
            # Inferenza
            predictions = model([video_input])[0]
            
            segs = predictions['segments'].cpu().numpy()
            scores = predictions['scores'].cpu().numpy()
            
            for j in range(len(segs)):
                t_start = float(segs[j][0])
                t_end = float(segs[j][1])
                score = float(scores[j])
                
                if t_end > t_start:
                    results.append({
                        'video_id': batch['video_id'],
                        't_start': round(t_start, 2),
                        't_end': round(t_end, 2),
                        'score': round(score, 4)
                    })
            
            if i % 50 == 0:
                print(f"   Processati {i}/{len(dataset)}...")

    df = pd.DataFrame(results)
    df.to_csv(output_path, index=False)
    print(f" Inferenza completata! Righe generate: {len(df)}")

# --- ESECUZIONE ---
# 1. Inizializza Modello
print(" Caricamento Architettura...")
model_768 = PtTransformer(**cfg_test).cuda()

# 2. Carica i Pesi dal path specifico
if os.path.exists(WEIGHTS_PATH):
    print(f" Caricamento pesi da: {WEIGHTS_PATH}")
    state = torch.load(WEIGHTS_PATH)
    # Pulizia chiavi (toglie 'module.' se c'è)
    clean_state = {k.replace('module.', ''): v for k, v in state.items()}
    model_768.load_state_dict(clean_state)
    
    # 3. Dataset Test (training=False)
    # Assicurati che SmartRandomCropDataset sia definito nel notebook!
    test_dataset = SmartRandomCropDataset(PATH_GT_CSV, PATH_EGO_DIR, PATH_OMNI_DIR, training=False)
    
    # 4. Lancia Inferenza
    run_final_inference(model_768, test_dataset, OUTPUT_CSV)
    
else:
    print(f" ERRORE: Il file pesi non esiste in: {WEIGHTS_PATH}")

🔧 Caricamento Architettura...
📥 Caricamento pesi da: /kaggle/input/best-model-loss-768/best_model_loss_768.pth
🔍 Mappatura file in corso (Ricorsiva)...
✅ Setup completato: 384 video pronti per il training.
🚀 Avvio Inferenza su 384 video...
💾 Output sarà salvato in: /kaggle/working/inferenza_finale_768.csv
   Processati 0/384...
   Processati 50/384...
   Processati 100/384...
   Processati 150/384...
   Processati 200/384...
   Processati 250/384...
   Processati 300/384...
   Processati 350/384...
✅ Inferenza completata! Righe generate: 28756


In [ ]:
# --- FUNZIONI DI EVALUATION ---
def calculate_iou_1d(pred_start, pred_end, gt_start, gt_end):
    start_max = max(pred_start, gt_start)
    end_min = min(pred_end, gt_end)
    intersection = max(0, end_min - start_max)
    pred_dur = pred_end - pred_start
    gt_dur = gt_end - gt_start
    union = pred_dur + gt_dur - intersection
    return intersection / union if union > 0 else 0.0

def evaluate_recall(gt_path, pred_path, thresholds=[0.1, 0.3, 0.5]):
    print("\n AVVIO VALUTAZIONE RECALL...")
    try:
        df_gt = pd.read_csv(gt_path)
        df_pred = pd.read_csv(pred_path)
    except Exception as e:
        print(f"Errore lettura file: {e}")
        return

    # Pulizia preliminare
    df_pred = df_pred[df_pred['t_end'] > df_pred['t_start']]
    
    # Raggruppamento per velocità
    preds_by_video = df_pred.groupby('video_id')
    total_gt = len(df_gt)
    matches_count = {t: 0 for t in thresholds}
    
    print(f" Confronto {len(df_pred)} predizioni con {total_gt} annotazioni reali...")

    for idx, gt_row in df_gt.iterrows():
        vid = gt_row['recording_id']
        gt_s, gt_e = gt_row['start_time'], gt_row['end_time']
        
        if vid not in preds_by_video.groups: continue
            
        # Prendi solo predizioni di quel video che si intersecano temporalmente
        vid_preds = preds_by_video.get_group(vid)
        candidates = vid_preds[
            (vid_preds['t_end'] > gt_s) & (vid_preds['t_start'] < gt_e)
        ]
        
        max_iou = 0.0
        for _, pred_row in candidates.iterrows():
            iou = calculate_iou_1d(pred_row['t_start'], pred_row['t_end'], gt_s, gt_e)
            if iou > max_iou: max_iou = iou
        
        for t in thresholds:
            if max_iou >= t: matches_count[t] += 1

    print("\n RISULTATI RECALL (Feature 768 + Soft-NMS):")
    print("-" * 50)
    for t in thresholds:
        recall = (matches_count[t] / total_gt) * 100
        print(f" Recall @ IoU {t} : {recall:.2f}%  ({matches_count[t]}/{total_gt})")
    print("-" * 50)

# --- ESECUZIONE TEST ---
evaluate_recall(PATH_GT_CSV, OUTPUT_CSV, thresholds=[0.1, 0.3, 0.5])


📊 AVVIO VALUTAZIONE RECALL...
🔄 Confronto 28756 predizioni con 5700 annotazioni reali...

🏆 RISULTATI RECALL (Feature 768 + Soft-NMS):
--------------------------------------------------
✅ Recall @ IoU 0.1 : 27.75%  (1582/5700)
✅ Recall @ IoU 0.3 : 22.05%  (1257/5700)
✅ Recall @ IoU 0.5 : 11.54%  (658/5700)
--------------------------------------------------
